# 93 datasets augmentation to find using suggested XE method's best delta_max (label for meta learning)

# Functions for Each Step

In [ ]:
import numpy as np
import pandas as pd 
import time
import os
import io
from scipy.io import arff

from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from llmfilter_dh import describe_data, run_filter_chunked
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)   # 5-fold-cross validation

import re
############################ Data feature names cleaning ############################
# This function cleans 'thoughts ?' -> 'thoughts?' and handles extra spaces
def clean_column_names(df):
    new_columns = []
    for col in df.columns:
        # 1. Remove leading/trailing whitespace
        c = col.strip()
        # 2. Replace multiple spaces with a single space
        c = re.sub(r'\s+', ' ', c)
        # 3. Remove space before punctuation (e.g., " ?" -> "?")
        c = re.sub(r'\s+([?.!,])', r'\1', c)
        new_columns.append(c)
    
    df.columns = new_columns
    return df

############################ Step 1: Identify Feature Types and Numericalize ############################
def identify_feature_types(df, t=15):
    metadata_list = []
    
    for col in df.columns:
        unique_vals = np.sort(df[col].unique())
        n_unique = len(unique_vals)
        dtype = df[col].dtype
        
        # --- Logic Gates for Categorization ---
        if n_unique <= t:
            # All low-cardinality features are treated as Categorical (Snapping required)
            category = 'Categ'
        
        else:
            if np.issubdtype(dtype, np.integer):
                # High-cardinality integers (Age, Study Hours)
                category = 'Num_disc'
            elif dtype == 'object':
                # High-cardinality strings (Names, IDs, Addresses)
                category = 'REMOVE'
            else:
                category = 'Num_cont' # Fallback for unknown numeric types
        
        metadata_list.append({
            'Feature': col,
            'Num Unique': n_unique,
            'Data Type': dtype,
            'Valid Values': unique_vals.tolist(),
            'Category': category
        })
    metadata = pd.DataFrame(metadata_list)
    Categ = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Categ']
    Num_disc = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_disc']
    Num_cont = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_cont']
    REMOVE = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'REMOVE']
        
    return metadata, Categ, Num_disc, Num_cont, REMOVE

def encode_to_numerical(df_org, REMOVE, Categ):
    # 1. Remove features in the REMOVE list
    df_cleaned = df_org.drop(REMOVE, axis=1)

    # 2. Initialize the translation dictionary (The "Rosetta Stone")
    # We will need this for Step 5 and Step 7
    mapping_dict = {}

    df_num = df_cleaned.copy()

    for col in Categ:
        # Get the valid values we stored in metadata
        # We sort them to ensure the mapping is consistent (e.g., 0 is always the same thing)
        unique_labels = sorted(df_num[col].unique())
        if col == Categ[-1]:
            # label = last column (always, Major = 0, Minor = 1)
            unique_labels = [df_num.iloc[:,-1].value_counts().index[0], df_num.iloc[:,-1].value_counts().index[1]]

        # Create the Forward (Text -> Num) and Reverse (Num -> Text) maps
        forward_map = {label: i for i, label in enumerate(unique_labels)}
        reverse_map = {i: label for i, label in enumerate(unique_labels)}

        # Save to our dictionary
        mapping_dict[col] = {
            'forward': forward_map,
            'reverse': reverse_map
        }

        # Apply the transformation
        df_num[col] = df_num[col].map(forward_map)

    # 3. Ensure all columns are numeric types (float or int) for SMOTE
    # This converts any remaining "object" types that might have slipped through
    df_num = df_num.apply(pd.to_numeric)
    
    return mapping_dict, df_num

############################ Step 2: Extra SMOTE generates new data points with Extrapolation ############################
def smote_generate(data, n_samples, delta_range=(0.0, 1.0), seed=None):
    # Set the seed for reproducibility if a number is provided
    # If seed is None, it uses the system clock (different every time)
    smote_k = 5                                                      # for knn in smote or extra smote
    rng = np.random.default_rng(seed)
    knn = NearestNeighbors(n_neighbors=smote_k).fit(data)            # Find k nearest neighbors for each minority point
    synthetic = []
    for _ in range(n_samples):
        idx = rng.integers(0, len(data))                             # Pick a random index from minority points
        origin = data[idx]                                           # the chosen minority point
        neighbor_distances, neighbor_indices = knn.kneighbors([origin])
        neighbor = data[rng.choice(neighbor_indices[0][1:])]         # Pick a random neighbor (excluding itself)  
        delta = rng.uniform(delta_range[0], delta_range[1])          # The random factor
        synthetic.append(origin + delta * (neighbor - origin))       # Formula: x_new = x_i + delta * (x_neighbor - x_i)
    return pd.DataFrame(synthetic, columns = df_num.columns[:-1])

############################ Step 3: Snap & Round (correction for original-like data) ############################
def process_extra_samples(df_raw_extra, Num_cont, Num_disc, Categ, meta_df):
    """
    Snaps and rounds raw synthetic data based on feature classifications.
    Input df_raw_extra is a DataFrame.
    """
    df_extra = df_raw_extra.copy()
    
    # FIX: Ensure 'Feature' is the index so we can look up by name
    # We check if 'Feature' is a column; if so, we set it as index.
    if 'Feature' in meta_df.columns:
        meta_df = meta_df.set_index('Feature')
    
    for col in df_extra.columns:
        if col in Num_cont:
            continue
            
        elif col in Num_disc:
            df_extra[col] = df_extra[col].round().astype(int)
            
        elif col in Categ:
            # Look up the number of categories
            # Note: Ensure this matches the column name in your metadata ('Num Unique')
            num_categories = meta_df.loc[col, 'Num Unique']
            
            # Valid options are the integer indices 0, 1, ..., n-1
            valid_indices = np.arange(num_categories)
            
            def snap_to_closest(val):
                # Using absolute difference to find the nearest integer label
                idx = (np.abs(valid_indices - val)).argmin()
                return valid_indices[idx]
            
            df_extra[col] = df_extra[col].apply(snap_to_closest).astype(int)

    return df_extra

############################ Step 4: ENN filtering ############################
def enn_filter(synthetic_points, orig_min, orig_maj):
    enn_k = 3                                                 # for knn in enn
    X_ref = np.vstack([orig_min, orig_maj])                   # Original whole data
    y_ref = np.array([1]*len(orig_min) + [0]*len(orig_maj))   # Original data's label
    knn = NearestNeighbors(n_neighbors=enn_k).fit(X_ref)     
    
    kept, removed = [], []
    for p in synthetic_points:                                # For each extra-generated point
        distances, indices = knn.kneighbors(p.reshape(1, -1)) # Find its neighbors' indices
        # If majority of neighbors in original data are minority class, KEEP.
        if np.sum(y_ref[indices[0]]) > (enn_k / 2):
            kept.append(p)
        else:
            removed.append(p)
    return pd.DataFrame(kept, columns = df_num.columns[:-1]), pd.DataFrame(removed, columns = df_num.columns[:-1])

# XE(Extra+ENN) - 5CV (cross-validation train datasets) - 56% * 5
## Extrapolation Generation + ENN filtering (removing) without LLM

In [ ]:
# Generation and removing with ENN (So, the factor needs to be 1~3)
needed_num_factor = 3

In [ ]:
# repeat the code below with different 'delta_value's
# {1.5, 2, 2.5, 3}, for extra smote, delta range = (1, delta_value)
delta_value = 1.5       

In [ ]:
# This code is only for data generation, so we don't handle test data (30% of the original) here.
# We use 5-fold cross-validation, so for 1 training, we need to generate 5 different datasets.
# df_num = original whole data (DataFrame), major label = 0, minor label = 1

for i in range(1, 98):
    start = time.time()
    if i == 23 or i == 81 or i == 82 or i == 84:        # duplicated
        continue
    # Load data
    df_org = pd.read_csv(r'/5. SMOTE_LLM/data_original/'+'ds'+ str(i) +'.csv')
    df_org = clean_column_names(df_org)
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    print('<Original Class>\n', df_org.iloc[:,-1].value_counts())
    # STEP1: Feature Types & Numericalize
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df_org, t=15)
    mapping_dict, df_num = encode_to_numerical(df_org, REMOVE, Categ)    # df_num: numericalized data    
    ##################### Original Data Info #######################
    print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
    print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
    X = df_val.iloc[:, :-1]
    y = df_val.iloc[:, -1]
    print('<Train-Data Class Dist>\n', y.value_counts())
    print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(y.value_counts()[1]/y.value_counts()[0]))
    ##################### For Validation Set #######################
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
    if i == 33:
        min_strategy = Strategy[1]                                      
    print("<min_strategy>:",min_strategy)

    for j in range(len(Strategy)):
        print("==========", "XE_{}".format(Strategy[j]), "==========")
        if min_strategy > Strategy[j]:
            continue         
        else:     
            n_iter=0
            for train_index, val_index in skf.split(X, y):
                n_iter += 1
                print("=======", "{}th-cv".format(n_iter), "=======")
                X_train = X.iloc[train_index]
                y_train = y.iloc[train_index]
                if n_iter == 1:
                    print(list(y_train).count(0), list(y_train).count(1), len(y_train))
                df_train = pd.concat([X_train, y_train], axis=1)        # Train Data (70% * 80% = 56%)
                majority_data = df_train[df_train.iloc[:,-1] == 0]      # Train majority data
                minority_data = df_train[df_train.iloc[:,-1] == 1]      # Train minority data
                needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
                print("Nedeed Samples:", needed_num)

                # Minority Generation
                majority_class = np.array(majority_data.iloc[:,:-1])    # majority data - features 
                minority_class = np.array(minority_data.iloc[:,:-1])    # minority data - features
                df_after_enn = pd.DataFrame()                       # LLM_kept_DataFrame  
                current_loop_seed = 0                                   # Start our seed at 0

                while len(df_after_enn) < needed_num:
                    n_to_gen = needed_num * needed_num_factor    # target generation (3~10 times of needed_num)

                    # step2. Generate with Extrapolation
                    df_raw_extra = smote_generate(minority_class, n_samples=n_to_gen, delta_range=(1, delta_value), 
                                               seed=current_loop_seed)   # Every repeat, it has different random seed
                    # step3. snap & round
                    df_extra = process_extra_samples(df_raw_extra, Num_cont, Num_disc, Categ, metadata) 

                    # step4. Remove with ENN
                    enn_kept, enn_removed = enn_filter(np.array(df_extra), minority_class, majority_class)
                    if len(enn_kept) > 0:
                        sub_df_after_enn = enn_kept.copy()
                        sub_df_after_enn[df_num.columns[-1]] = 1
                        df_after_enn = pd.concat([df_after_enn, sub_df_after_enn], axis=0)      

                    print(f"Random Seed {current_loop_seed} complete. Total Kept: {len(df_after_enn)}/{needed_num}")    
                    current_loop_seed += 1

                # Final Result
                new_data = df_after_enn.iloc[:needed_num].reset_index(drop=True)
                over_df = pd.concat([df_train, new_data], axis=0)
                over_df.to_csv("Generated Data/XE/(delta={})ds{}_XE_{}_{}th.csv".format(delta_value, i, Strategy[j], n_iter), index=False)

# XE - Train dataset 70%

In [ ]:
# repeat the code below with different 'delta_value's
# {1.5, 2, 2.5, 3}, for extra smote, delta range = (1, delta_value)
delta_value = 1.5

In [ ]:
for i in range(1, 98):
    start = time.time()
    if i == 23 or i == 81 or i == 82 or i == 84:        # duplicated
        continue
    # Load data
    df_org = pd.read_csv(r'/5. SMOTE_LLM/data_original/'+'ds'+ str(i) +'.csv')
    df_org = clean_column_names(df_org)
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    print('<Original Class>\n', df_org.iloc[:,-1].value_counts())
    # STEP1: Feature Types & Numericalize
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df_org, t=15)
    mapping_dict, df_num = encode_to_numerical(df_org, REMOVE, Categ)    # df_num: numericalized data    
    ##################### Original Data Info #######################
    print('<Whole-Data Class Dist>\n', df_num.iloc[:,-1].value_counts())
    print('<Whole-Data IMB ratio>\n', "1:{: .4f}".format(df_num.iloc[:,-1].value_counts()[1]/df_num.iloc[:,-1].value_counts()[0]))

    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df_num, test_size=0.3, random_state=100, stratify=df_num.iloc[:,-1])
    majority_data = df_val[df_val.iloc[:,-1] == 0]      # Train majority data
    minority_data = df_val[df_val.iloc[:,-1] == 1]      # Train minority data
    print('<Train-Data Class Dist>\n', df_val.iloc[:,-1].value_counts())
    print('<Train-Data IMB ratio>\n', "1:{: .4f}".format(df_val.iloc[:,-1].value_counts()[1]/df_val.iloc[:,-1].value_counts()[0]))
    ##################### For Validation Set #######################
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]                              # Minority Class Target Ratio     
    ind = int((df_val.iloc[:,-1].value_counts()[1]/df_val.iloc[:,-1].value_counts()[0])//0.2)
    min_strategy = Strategy[ind]                                      # Minimum Target Ratio for validation set (70%)
    if i == 33:
        min_strategy = Strategy[1]                                      
    print("<min_strategy>:",min_strategy)

    for j in range(len(Strategy)):
        print("==========", "XE_{}".format(Strategy[j]), "==========")
        if min_strategy > Strategy[j]:
            continue         
        else:     
            needed_num = int(len(majority_data) * Strategy[j]) - len(minority_data)   # Number Gneration Needed
            print("Nedeed Samples:", needed_num)

            # Minority Generation
            majority_class = np.array(majority_data.iloc[:,:-1])    # majority data - features 
            minority_class = np.array(minority_data.iloc[:,:-1])    # minority data - features
            df_after_enn = pd.DataFrame()                       # LLM_kept_DataFrame  
            current_loop_seed = 0                                   # Start our seed at 0

            while len(df_after_enn) < needed_num:
                n_to_gen = needed_num * needed_num_factor    # target generation (3~10 times of needed_num)

                # step2. Generate with Extrapolation
                df_raw_extra = smote_generate(minority_class, n_samples=n_to_gen, delta_range=(1, delta_value), 
                                           seed=current_loop_seed)   # Every repeat, it has different random seed
                # step3. snap & round
                df_extra = process_extra_samples(df_raw_extra, Num_cont, Num_disc, Categ, metadata) 

                # step4. Remove with ENN
                enn_kept, enn_removed = enn_filter(np.array(df_extra), minority_class, majority_class)
                if len(enn_kept) > 0:
                    sub_df_after_enn = enn_kept.copy()
                    sub_df_after_enn[df_num.columns[-1]] = 1
                    df_after_enn = pd.concat([df_after_enn, sub_df_after_enn], axis=0)      

                current_loop_seed += 1

            # Final Result
            new_data = df_after_enn.iloc[:needed_num].reset_index(drop=True)
            over_df = pd.concat([df_val, new_data], axis=0)
            over_df.to_csv("Generated Data/XE/(delta={})ds{}_XE_{}_full.csv".format(delta_value, i, Strategy[j]), index=False)